# 02 — Inline claim-based authz

Same shared API key as pattern 1, but now the agent does *something* with
the user's identity: it fetches the user's JWT, reads the claims out of it,
and constructs scoped tool calls based on what the claims say.

The tool still has no idea who's asking (it just sees `X-API-Key`), but
at least the call going out is narrower than "give me everything."

## The gap from pattern 1

In pattern 1 the agent had no information to act on. The LLM was the only
thing trying to keep alice from seeing carlo's data — and the LLM is not
a security boundary.

This pattern fixes the *information* problem: the agent now knows the
user's role, department, and reporting structure. It uses that information
to scope the call before it goes out. Authz is still done in the agent,
but it's done with structured data instead of just prompt-following.

In [ ]:
from shared.agent import Agent
from shared.auth import InlineClaimAuth, fetch_user_jwt, decode_jwt
from shared.tools import ALL_TOOLS
from shared.display import run_as, show_what_tool_saw, show_token
from shared.config import EXPENSE_SERVICE_URL

strategy = InlineClaimAuth()
agent = Agent(strategy=strategy, tools=ALL_TOOLS)
print(f"strategy: {strategy.name}")

## What's in the JWT the agent reads

Before running the agent, let's look at what the agent will see when it
fetches alice's user JWT. The custom claims on this token are the
*input* the agent uses to make scoping decisions.

In [ ]:
alice_jwt = fetch_user_jwt("alice")
show_token(alice_jwt, label="alice user JWT (what InlineClaimAuth reads)")

## Same prompt, three users

Watch how the agent scopes the tool calls differently for each user — but
also watch where the scoping breaks down.

In [ ]:
result_alice = run_as("alice", "What expenses do I have?", agent)

In [ ]:
# Right after alice's run, peek at what the service saw.
show_what_tool_saw(EXPENSE_SERVICE_URL)

In [ ]:
result_bob = run_as("bob", "What expenses do my team have?", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

In [ ]:
result_carlo = run_as("carlo", "Show me all the expenses across the company.", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

## Tradeoffs

- **Where authz lives:** in the agent. Specifically, in `InlineClaimAuth`'s
  `prepare()` method, which has hand-coded rules for which query parameters
  to add for which roles. Look at the source — every new role, every new
  tool, every new field requires editing that code.
- **Tool sees real user:** still no. The service still sees `method=api_key`.
  The narrowing happens *before* the call goes out, not on the service side.
- **Cryptographic proof of identity:** still none.
- **Least privilege:** partial. The agent can narrow what it asks for, but
  it could ask for anything if it wanted to (or if it had a bug).
- **The gap that breaks the demo for alice:** look at what happened above.
  The agent could narrow the call for bob (a manager — added
  `?department=engineering`) and didn't need to narrow for carlo (admin).
  But for alice the agent had no useful query parameter to add — the
  service doesn't accept `?user_id=alice` from an api-key caller, so the
  agent ended up sending an unfiltered call. Alice's "expenses" therefore
  include everyone's. **This is the pedagogical point**: doing authz in
  the agent only works as well as the API surface lets you express it.
- **Audit trail:** logs still say "agent called expense-service with the
  api key". The fact that the agent *internally* knew it was acting for
  alice is invisible to the service.

Real codebases that do this kind of inline scoping tend to grow a thicket
of role-checks scattered across the codebase. Every tool has slightly
different rules, every new feature requires touching the auth logic, and
review burden goes up linearly with feature count.

## What's still missing?

We've shown the agent can use claims to narrow some calls. But:

1. The logic lives in agent code, not in policy. Adding a new tool means
   editing `InlineClaimAuth`.
2. The narrowing only works as far as the tool's API surface allows.
   Alice's case is the proof: there was nothing useful to narrow with.
3. The tool still trusts whoever has the API key.

The next notebook (`03_external_authz_agent`) keeps the same shape — agent
has the JWT, tool gets the API key — but moves the policy logic out of
agent code into a centralized policy engine (OPA). This fixes problem (1):
the rules become declarative and auditable. It does **not** fix problems
(2) and (3); those are what patterns 4-8 are for.